In [106]:
import pandas as pd
import re
import numpy as np
from datetime import datetime, timedelta
import random
import unicodedata

In [107]:
# Đọc file và gán loại BĐS
files = {
    'nd-hcm.xlsx': 'Nhà Đất Thổ Cư',
    'nd-hn.xlsx': 'Nhà Đất Thổ Cư',
    'nd-bd.xlsx': 'Nhà Đất Thổ Cư',
    'nd-dn.xlsx': 'Nhà Đất Thổ Cư',
    'cc-hcm.xlsx': 'Căn Hộ Chung Cư',
    'cc-hn.xlsx': 'Căn Hộ Chung Cư'
}

dfs = []

for file, loai_bds in files.items():
    dfr1 = pd.read_excel(file)
    dfr1['Loại BĐS'] = loai_bds
    dfs.append(dfr1)

# Gộp tất cả thành một DataFrame
df1 = pd.concat(dfs, ignore_index=True)
df1

,Tên dự án,Giá,Diện tích,Vị trí,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS
0,NHÀ NGUYỄN XIỂN - QUẬN 9 - 654M2 - NGANG 26M -...,"18,2 tỷ",654 m²,"Quận 9, Hồ Chí Minh",4 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư
1,"SIÊU PHẨM ĐẦU TƯ NHÀ ĐẤT THỦ ĐỨC 750M2, MT 23M...",17 tỷ,750 m²,"Thủ Đức, Hồ Chí Minh",15 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư
2,"BÁN NHÀ 3 MẶT, ĐƯỜNG TX22, PHƯỜNG THẠNH XUÂN, ...",16 tỷ,373 m²,"Quận 12, Hồ Chí Minh",NaN,NaN,NaN,Nhà Đất Thổ Cư
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,"2,45 tỷ",61 m²,"Quận 3, Hồ Chí Minh",4 Phòng ngủ,3 WC,NaN,Nhà Đất Thổ Cư
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,"4,95 tỷ",50 m²,"Quận 2, Hồ Chí Minh",1 Phòng ngủ,1 WC,NaN,Nhà Đất Thổ Cư
...,...,...,...,...,...,...,...,...
71227,"Chính chủ cần bán gấp căn hộ 4PN, 141m2, căn g...",Giá thỏa thuận,141 m²,"Thanh Trì, Hà Nội",4 Phòng ngủ,3 WC,NaN,Căn Hộ Chung Cư
71228,Chính chủ cần bán gấp căn 3 ngủ FLC Cầu Giấy g...,"7,5 tỷ",98 m²,"Cầu Giấy, Hà Nội",3 Phòng ngủ,2 WC,NaN,Căn Hộ Chung Cư
71229,"Bán CC 3PN 2WC tại AZ Lâm Viên Complex, giá 9,...","9,7 tỷ",129 m²,"Cầu Giấy, Hà Nội",3 Phòng ngủ,2 WC,NaN,Căn Hộ Chung Cư
71230,"Cần bán CHCC Hòa Bình Green Apartment, Vĩnh Ph...","6,75 tỷ",90 m²,"Ba Đình, Hà Nội",2 Phòng ngủ,2 WC,NaN,Căn Hộ Chung Cư


In [108]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71232 entries, 0 to 71231
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Tên dự án    71232 non-null  object 
 1   Giá          71232 non-null  object 
 2   Diện tích    71232 non-null  object 
 3   Vị trí       71232 non-null  object 
 4   Phòng ngủ    61954 non-null  object 
 5   Nhà vệ sinh  58440 non-null  object 
 6   Ngày đăng    0 non-null      float64
 7   Loại BĐS     71232 non-null  object 
dtypes: float64(1), object(7)
memory usage: 4.3+ MB


In [109]:
# Tách quận/huyện từ cột "Vị trí" (giả sử định dạng "Quận/Huyện, Tỉnh/Thành phố")
df1['Quận/Huyện'] = df1['Vị trí'].str.split(',').str[0].str.strip()

# Tách tỉnh/thành phố từ cột "Vị trí"
df1['Tỉnh/Thành phố'] = df1['Vị trí'].str.split(',').str[1].str.strip()

# Xoá cột "Vị trí" sau khi đã tách
df1.drop(columns=['Vị trí'], inplace=True)

In [110]:
df1

,Tên dự án,Giá,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố
0,NHÀ NGUYỄN XIỂN - QUẬN 9 - 654M2 - NGANG 26M -...,"18,2 tỷ",654 m²,4 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư,Quận 9,Hồ Chí Minh
1,"SIÊU PHẨM ĐẦU TƯ NHÀ ĐẤT THỦ ĐỨC 750M2, MT 23M...",17 tỷ,750 m²,15 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư,Thủ Đức,Hồ Chí Minh
2,"BÁN NHÀ 3 MẶT, ĐƯỜNG TX22, PHƯỜNG THẠNH XUÂN, ...",16 tỷ,373 m²,NaN,NaN,NaN,Nhà Đất Thổ Cư,Quận 12,Hồ Chí Minh
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,"2,45 tỷ",61 m²,4 Phòng ngủ,3 WC,NaN,Nhà Đất Thổ Cư,Quận 3,Hồ Chí Minh
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,"4,95 tỷ",50 m²,1 Phòng ngủ,1 WC,NaN,Nhà Đất Thổ Cư,Quận 2,Hồ Chí Minh
...,...,...,...,...,...,...,...,...,...
71227,"Chính chủ cần bán gấp căn hộ 4PN, 141m2, căn g...",Giá thỏa thuận,141 m²,4 Phòng ngủ,3 WC,NaN,Căn Hộ Chung Cư,Thanh Trì,Hà Nội
71228,Chính chủ cần bán gấp căn 3 ngủ FLC Cầu Giấy g...,"7,5 tỷ",98 m²,3 Phòng ngủ,2 WC,NaN,Căn Hộ Chung Cư,Cầu Giấy,Hà Nội
71229,"Bán CC 3PN 2WC tại AZ Lâm Viên Complex, giá 9,...","9,7 tỷ",129 m²,3 Phòng ngủ,2 WC,NaN,Căn Hộ Chung Cư,Cầu Giấy,Hà Nội
71230,"Cần bán CHCC Hòa Bình Green Apartment, Vĩnh Ph...","6,75 tỷ",90 m²,2 Phòng ngủ,2 WC,NaN,Căn Hộ Chung Cư,Ba Đình,Hà Nội


In [111]:
# Đọc file và gán tỉnh/thành phố
files1 = {
    'Nha_dat_BinhDuong.csv' : 'Bình Dương',
    'Nha_dat_DaNang.csv' : 'Đà Nẵng',
    'Nha_dat_DongNai.csv': 'Đồng Nai',
    'Nha_dat_HaNoi.csv' : 'Hà Nội',
    'Nha_dat_HCM.csv' : 'Hồ Chí Minh',
}

# Đọc tất cả các file CSV và gộp chúng lại
dfs1 = []

for file, tinh in files1.items():
    dfr2 = pd.read_csv(file)
    dfr2['Tỉnh/Thành phố'] = tinh
    dfs1.append(dfr2)

# Gộp tất cả thành một DataFrame
df2 = pd.concat(dfs1, ignore_index=True)
df2

,Tên dự án,Giá,Diện tích,Loại nhà,Vị trí,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Trang,Tỉnh/Thành phố
0,CĂN HỘ THE EMERALD 68 MỞ BÁN GIỎ HÀNG ĐỘC QUYỀ...,2.6 Tỷ,48 M²,Căn Hộ Chung Cư,Thành Phố Thuận An,2 Phòng ngủ,1 WC,"Hôm nay, 17 phút trước.",1,Bình Dương
1,CHÍNH CHỦ BÁN LỖ CĂN C.05.01 NEW GALAXY LÀNG Đ...,1.1 Tỷ,50 M²,Căn Hộ Chung Cư,New Galaxy Hưng Thịnh,1 Phòng ngủ,1 WC,"Hôm nay, 17 phút trước.",1,Bình Dương
2,CẬP NHẬT BẢNG GIÁ ĐỢT MỞ BÁN MỚI CĂN HỘ THE EM...,3.1 Tỷ,67 M²,Căn Hộ Chung Cư,Thành Phố Thuận An,2 Phòng ngủ,2 WC,"Hôm nay, 17 phút trước.",1,Bình Dương
3,CẦN BÁN ĐẤT KHU DÂN ĐÔNG 656M2 THỔ CƯ 300M2 SÁ...,495 Triệu,656 M²,Nhà Đất Thổ Cư,Huyện Dầu Tiếng,NaN,NaN,"Hôm nay, 1 giờ 20 phút trước.",1,Bình Dương
4,NHÀ MỚI 1 TRỆT 2 LẦU GẦN TRUNG TÂM THỦ DẦU MỘT...,3.35 Tỷ,360 M²,Nhà Mặt Phố,Thành Phố Thủ Dầu Một,3 Phòng ngủ,4 WC,"Hôm nay, 1 giờ 42 phút trước.",1,Bình Dương
...,...,...,...,...,...,...,...,...,...,...
291976,MT NGUYỄN MINH CHÂU 4.5 X 13.5M NHÀ 4 TẦNG BTC...,8.3 Tỷ,60 M²,Nhà Mặt Phố,Nguyễn Minh Châu,4 Phòng ngủ,4 WC,"Hôm nay, 29 phút trước.",3208,Hồ Chí Minh
291977,NHÀ MỚI QUẬN 10 40M² FULL NỘI THẤT 8.2 TỶ - GẦ...,8.2 Tỷ,37 M²,Nhà Trong Ngõ,Ngô Gia Tự,4 Phòng ngủ,4 WC,"Hôm nay, 8 phút trước.",3209,Hồ Chí Minh
291978,BÁN NHÀ CĂN GÓC 5 TẦNG MẶT TIỀN KINH DOANH ĐƯỜ...,29 Tỷ,88 M²,Nhà Mặt Phố,Quận 6,9 Phòng ngủ,9 WC,"Hôm nay, 22 phút trước.",3209,Hồ Chí Minh
291979,BÁN KHÁCH SẠN ĐANG KD 21 PHÒNG TẠI P2 QUẬN 6,30 Tỷ,100 M²,Nhà Mặt Phố,Quận 6,11 Phòng ngủ,11 WC,"Hôm nay, 21 phút trước.",3209,Hồ Chí Minh


In [112]:
# Đổi tên cột "Vị trí" thành "Quận/Huyện"
df2.rename(columns={'Vị trí': 'Quận/Huyện', 'Loại nhà' : 'Loại BĐS'}, inplace=True)

In [113]:
# sắp xếp lại thứ tự cột cho trùng với df1
df2 = df2[['Tên dự án', 'Giá', 'Diện tích', 'Phòng ngủ', 'Nhà vệ sinh', 'Ngày đăng', 'Loại BĐS', 'Quận/Huyện', 'Tỉnh/Thành phố']]
df2

,Tên dự án,Giá,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố
0,CĂN HỘ THE EMERALD 68 MỞ BÁN GIỎ HÀNG ĐỘC QUYỀ...,2.6 Tỷ,48 M²,2 Phòng ngủ,1 WC,"Hôm nay, 17 phút trước.",Căn Hộ Chung Cư,Thành Phố Thuận An,Bình Dương
1,CHÍNH CHỦ BÁN LỖ CĂN C.05.01 NEW GALAXY LÀNG Đ...,1.1 Tỷ,50 M²,1 Phòng ngủ,1 WC,"Hôm nay, 17 phút trước.",Căn Hộ Chung Cư,New Galaxy Hưng Thịnh,Bình Dương
2,CẬP NHẬT BẢNG GIÁ ĐỢT MỞ BÁN MỚI CĂN HỘ THE EM...,3.1 Tỷ,67 M²,2 Phòng ngủ,2 WC,"Hôm nay, 17 phút trước.",Căn Hộ Chung Cư,Thành Phố Thuận An,Bình Dương
3,CẦN BÁN ĐẤT KHU DÂN ĐÔNG 656M2 THỔ CƯ 300M2 SÁ...,495 Triệu,656 M²,NaN,NaN,"Hôm nay, 1 giờ 20 phút trước.",Nhà Đất Thổ Cư,Huyện Dầu Tiếng,Bình Dương
4,NHÀ MỚI 1 TRỆT 2 LẦU GẦN TRUNG TÂM THỦ DẦU MỘT...,3.35 Tỷ,360 M²,3 Phòng ngủ,4 WC,"Hôm nay, 1 giờ 42 phút trước.",Nhà Mặt Phố,Thành Phố Thủ Dầu Một,Bình Dương
...,...,...,...,...,...,...,...,...,...
291976,MT NGUYỄN MINH CHÂU 4.5 X 13.5M NHÀ 4 TẦNG BTC...,8.3 Tỷ,60 M²,4 Phòng ngủ,4 WC,"Hôm nay, 29 phút trước.",Nhà Mặt Phố,Nguyễn Minh Châu,Hồ Chí Minh
291977,NHÀ MỚI QUẬN 10 40M² FULL NỘI THẤT 8.2 TỶ - GẦ...,8.2 Tỷ,37 M²,4 Phòng ngủ,4 WC,"Hôm nay, 8 phút trước.",Nhà Trong Ngõ,Ngô Gia Tự,Hồ Chí Minh
291978,BÁN NHÀ CĂN GÓC 5 TẦNG MẶT TIỀN KINH DOANH ĐƯỜ...,29 Tỷ,88 M²,9 Phòng ngủ,9 WC,"Hôm nay, 22 phút trước.",Nhà Mặt Phố,Quận 6,Hồ Chí Minh
291979,BÁN KHÁCH SẠN ĐANG KD 21 PHÒNG TẠI P2 QUẬN 6,30 Tỷ,100 M²,11 Phòng ngủ,11 WC,"Hôm nay, 21 phút trước.",Nhà Mặt Phố,Quận 6,Hồ Chí Minh


In [114]:
# Đọc file và gán tỉnh/thành phố
files2 = {
    'Nha_dat_BinhDuong.json' : 'Bình Dương',
    'Nha_dat_DaNang.json' : 'Đà Nẵng',
    'Nha_dat_DongNai.json': 'Đồng Nai',
    'Nha_dat_HaNoi.json' : 'Hà Nội',
    'Nha_dat_HCM.json' : 'Hồ Chí Minh',
    'Nha_dat_HN_total_711.json' : 'Hà Nội'
}

# Đọc tất cả các file CSV và gộp chúng lại
dfs2 = []

for file, tinh in files2.items():
    dfr3 = pd.read_json(file)
    dfr3['Tỉnh/Thành phố'] = tinh
    dfs2.append(dfr3)

# Gộp tất cả thành một DataFrame
df3 = pd.concat(dfs2, ignore_index=True)
df3

,Tên dự án,Giá,Diện tích,Loại nhà,Vị trí,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Trang,Tỉnh/Thành phố
0,CĂN HỘ THE EMERALD 68 MỞ BÁN GIỎ HÀNG ĐỘC QUYỀ...,2.6 Tỷ,48 M²,Căn Hộ Chung Cư,Thành Phố Thuận An,2 Phòng ngủ,1 WC,"Hôm nay, 17 phút trước.",1,Bình Dương
1,CHÍNH CHỦ BÁN LỖ CĂN C.05.01 NEW GALAXY LÀNG Đ...,1.1 Tỷ,50 M²,Căn Hộ Chung Cư,New Galaxy Hưng Thịnh,1 Phòng ngủ,1 WC,"Hôm nay, 17 phút trước.",1,Bình Dương
2,CẬP NHẬT BẢNG GIÁ ĐỢT MỞ BÁN MỚI CĂN HỘ THE EM...,3.1 Tỷ,67 M²,Căn Hộ Chung Cư,Thành Phố Thuận An,2 Phòng ngủ,2 WC,"Hôm nay, 17 phút trước.",1,Bình Dương
3,CẦN BÁN ĐẤT KHU DÂN ĐÔNG 656M2 THỔ CƯ 300M2 SÁ...,495 Triệu,656 M²,Nhà Đất Thổ Cư,Huyện Dầu Tiếng,,,"Hôm nay, 1 giờ 20 phút trước.",1,Bình Dương
4,NHÀ MỚI 1 TRỆT 2 LẦU GẦN TRUNG TÂM THỦ DẦU MỘT...,3.35 Tỷ,360 M²,Nhà Mặt Phố,Thành Phố Thủ Dầu Một,3 Phòng ngủ,4 WC,"Hôm nay, 1 giờ 42 phút trước.",1,Bình Dương
...,...,...,...,...,...,...,...,...,...,...
251166,"BÁN TÒA NHÀ MẶT PHỐ THỢ NHUỘM, 230M 9 TẦNG CÓ ...",192 Tỷ,230 M²,Nhà Mặt Phố,Quận Hoàn Kiếm,11 Phòng ngủ,10 WC,"15/6/2024, lúc: 14 giờ 21 phút",2470,Hà Nội
251167,"BÁN NHÀ NGỌC KHÁNH VỊ TRÍ HIẾM ĐẸP, 300M2 MẶT ...",76 Tỷ,300 M²,Nhà Trong Ngõ,Quận Ba Đình,2 Phòng ngủ,2 WC,"15/6/2024, lúc: 14 giờ 21 phút",2470,Hà Nội
251168,"BÁN BIỆT THỰ NAM TRUNG YÊN, 180M MẶT TIỀN 12M ...",56 Tỷ,180 M²,"Nhà Biệt Thự, Liền Kề",Quận Cầu Giấy,,,"15/6/2024, lúc: 14 giờ 21 phút",2471,Hà Nội
251169,-B.Á.N NHÀ HOÀNG MAI - NHÀ ĐẸP LONG LANH- Ô TÔ...,6.8 Tỷ,40 M²,Nhà Đất Thổ Cư,Quận Hoàng Mai,,,"15/6/2024, lúc: 14 giờ 11 phút",2471,Hà Nội


In [115]:
# Đổi tên cột "Vị trí" thành "Quận/Huyện"
df3.rename(columns={'Vị trí': 'Quận/Huyện', 'Loại nhà' : 'Loại BĐS'}, inplace=True)

In [116]:
# sắp xếp lại thứ tự cột cho trùng với df1
df3 = df3[['Tên dự án', 'Giá', 'Diện tích', 'Phòng ngủ', 'Nhà vệ sinh', 'Ngày đăng', 'Loại BĐS', 'Quận/Huyện', 'Tỉnh/Thành phố']]
df3

,Tên dự án,Giá,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố
0,CĂN HỘ THE EMERALD 68 MỞ BÁN GIỎ HÀNG ĐỘC QUYỀ...,2.6 Tỷ,48 M²,2 Phòng ngủ,1 WC,"Hôm nay, 17 phút trước.",Căn Hộ Chung Cư,Thành Phố Thuận An,Bình Dương
1,CHÍNH CHỦ BÁN LỖ CĂN C.05.01 NEW GALAXY LÀNG Đ...,1.1 Tỷ,50 M²,1 Phòng ngủ,1 WC,"Hôm nay, 17 phút trước.",Căn Hộ Chung Cư,New Galaxy Hưng Thịnh,Bình Dương
2,CẬP NHẬT BẢNG GIÁ ĐỢT MỞ BÁN MỚI CĂN HỘ THE EM...,3.1 Tỷ,67 M²,2 Phòng ngủ,2 WC,"Hôm nay, 17 phút trước.",Căn Hộ Chung Cư,Thành Phố Thuận An,Bình Dương
3,CẦN BÁN ĐẤT KHU DÂN ĐÔNG 656M2 THỔ CƯ 300M2 SÁ...,495 Triệu,656 M²,,,"Hôm nay, 1 giờ 20 phút trước.",Nhà Đất Thổ Cư,Huyện Dầu Tiếng,Bình Dương
4,NHÀ MỚI 1 TRỆT 2 LẦU GẦN TRUNG TÂM THỦ DẦU MỘT...,3.35 Tỷ,360 M²,3 Phòng ngủ,4 WC,"Hôm nay, 1 giờ 42 phút trước.",Nhà Mặt Phố,Thành Phố Thủ Dầu Một,Bình Dương
...,...,...,...,...,...,...,...,...,...
251166,"BÁN TÒA NHÀ MẶT PHỐ THỢ NHUỘM, 230M 9 TẦNG CÓ ...",192 Tỷ,230 M²,11 Phòng ngủ,10 WC,"15/6/2024, lúc: 14 giờ 21 phút",Nhà Mặt Phố,Quận Hoàn Kiếm,Hà Nội
251167,"BÁN NHÀ NGỌC KHÁNH VỊ TRÍ HIẾM ĐẸP, 300M2 MẶT ...",76 Tỷ,300 M²,2 Phòng ngủ,2 WC,"15/6/2024, lúc: 14 giờ 21 phút",Nhà Trong Ngõ,Quận Ba Đình,Hà Nội
251168,"BÁN BIỆT THỰ NAM TRUNG YÊN, 180M MẶT TIỀN 12M ...",56 Tỷ,180 M²,,,"15/6/2024, lúc: 14 giờ 21 phút","Nhà Biệt Thự, Liền Kề",Quận Cầu Giấy,Hà Nội
251169,-B.Á.N NHÀ HOÀNG MAI - NHÀ ĐẸP LONG LANH- Ô TÔ...,6.8 Tỷ,40 M²,,,"15/6/2024, lúc: 14 giờ 11 phút",Nhà Đất Thổ Cư,Quận Hoàng Mai,Hà Nội


In [117]:
# Gộp 3 DataFrame df1 và df2
df = pd.concat([df1, df2, df3], ignore_index=True)
df

,Tên dự án,Giá,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố
0,NHÀ NGUYỄN XIỂN - QUẬN 9 - 654M2 - NGANG 26M -...,"18,2 tỷ",654 m²,4 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư,Quận 9,Hồ Chí Minh
1,"SIÊU PHẨM ĐẦU TƯ NHÀ ĐẤT THỦ ĐỨC 750M2, MT 23M...",17 tỷ,750 m²,15 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư,Thủ Đức,Hồ Chí Minh
2,"BÁN NHÀ 3 MẶT, ĐƯỜNG TX22, PHƯỜNG THẠNH XUÂN, ...",16 tỷ,373 m²,NaN,NaN,NaN,Nhà Đất Thổ Cư,Quận 12,Hồ Chí Minh
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,"2,45 tỷ",61 m²,4 Phòng ngủ,3 WC,NaN,Nhà Đất Thổ Cư,Quận 3,Hồ Chí Minh
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,"4,95 tỷ",50 m²,1 Phòng ngủ,1 WC,NaN,Nhà Đất Thổ Cư,Quận 2,Hồ Chí Minh
...,...,...,...,...,...,...,...,...,...
614379,"BÁN TÒA NHÀ MẶT PHỐ THỢ NHUỘM, 230M 9 TẦNG CÓ ...",192 Tỷ,230 M²,11 Phòng ngủ,10 WC,"15/6/2024, lúc: 14 giờ 21 phút",Nhà Mặt Phố,Quận Hoàn Kiếm,Hà Nội
614380,"BÁN NHÀ NGỌC KHÁNH VỊ TRÍ HIẾM ĐẸP, 300M2 MẶT ...",76 Tỷ,300 M²,2 Phòng ngủ,2 WC,"15/6/2024, lúc: 14 giờ 21 phút",Nhà Trong Ngõ,Quận Ba Đình,Hà Nội
614381,"BÁN BIỆT THỰ NAM TRUNG YÊN, 180M MẶT TIỀN 12M ...",56 Tỷ,180 M²,,,"15/6/2024, lúc: 14 giờ 21 phút","Nhà Biệt Thự, Liền Kề",Quận Cầu Giấy,Hà Nội
614382,-B.Á.N NHÀ HOÀNG MAI - NHÀ ĐẸP LONG LANH- Ô TÔ...,6.8 Tỷ,40 M²,,,"15/6/2024, lúc: 14 giờ 11 phút",Nhà Đất Thổ Cư,Quận Hoàng Mai,Hà Nội


In [118]:
# Chỉ giữ lại các loại BĐS là "Căn Hộ Chung Cư" hoặc "Nhà Đất Thổ Cư"
df = df[df['Loại BĐS'].str.lower().isin(['căn hộ chung cư', 'nhà đất thổ cư'])]
df

,Tên dự án,Giá,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố
0,NHÀ NGUYỄN XIỂN - QUẬN 9 - 654M2 - NGANG 26M -...,"18,2 tỷ",654 m²,4 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư,Quận 9,Hồ Chí Minh
1,"SIÊU PHẨM ĐẦU TƯ NHÀ ĐẤT THỦ ĐỨC 750M2, MT 23M...",17 tỷ,750 m²,15 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư,Thủ Đức,Hồ Chí Minh
2,"BÁN NHÀ 3 MẶT, ĐƯỜNG TX22, PHƯỜNG THẠNH XUÂN, ...",16 tỷ,373 m²,NaN,NaN,NaN,Nhà Đất Thổ Cư,Quận 12,Hồ Chí Minh
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,"2,45 tỷ",61 m²,4 Phòng ngủ,3 WC,NaN,Nhà Đất Thổ Cư,Quận 3,Hồ Chí Minh
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,"4,95 tỷ",50 m²,1 Phòng ngủ,1 WC,NaN,Nhà Đất Thổ Cư,Quận 2,Hồ Chí Minh
...,...,...,...,...,...,...,...,...,...
614376,48M2 ĐA TỐN NGÕ 2.5M GIÁ CHỈ 1.95 TỶ BỚT NHIỀU...,1.95 Tỷ,48 M²,,,"15/6/2024, lúc: 14 giờ 21 phút",Nhà Đất Thổ Cư,Huyện Gia Lâm,Hà Nội
614377,-B.Á.N NHÀ BẠCH MAI -- NHÀ ĐẸP LONG LANH- FULL...,4.8 Tỷ,42 M²,,,"15/6/2024, lúc: 14 giờ 20 phút",Nhà Đất Thổ Cư,Quận Hai Bà Trưng,Hà Nội
614378,👉-B.Á.N ĐỊNH CÔNG THƯỢNG -DT:48MX4TẦNG - GIÁ 6...,6.6 Tỷ,48 M²,,,"15/6/2024, lúc: 14 giờ 16 phút",Nhà Đất Thổ Cư,Quận Hoàng Mai,Hà Nội
614382,-B.Á.N NHÀ HOÀNG MAI - NHÀ ĐẸP LONG LANH- Ô TÔ...,6.8 Tỷ,40 M²,,,"15/6/2024, lúc: 14 giờ 11 phút",Nhà Đất Thổ Cư,Quận Hoàng Mai,Hà Nội


In [119]:
# Hàm để trích xuất năm từ chuỗi ngày
def process_ngay_dang(value):
    if pd.isna(value) or '/' not in str(value):
        # Random ngày từ 1/1/2025 đến hiện tại
        start_date = datetime(2025, 1, 1)
        end_date = datetime.now()
        random_date = start_date + timedelta(days=random.randint(0, (end_date - start_date).days))
        return random_date.date()
    else:
        # Tách lấy phần trước dấu phẩy và parse thành datetime
        try:
            date_part = value.split(',')[0].strip()
            parsed_date = pd.to_datetime(date_part, dayfirst=True, errors='coerce')
            return parsed_date.date() if pd.notna(parsed_date) else np.nan
        except:
            return np.nan

# Tạo cột mới đã xử lý
df['Thời gian đăng'] = df['Ngày đăng'].apply(process_ngay_dang)
df['Thời gian đăng'] = pd.to_datetime(df['Thời gian đăng'], errors='coerce')
df

C:\Users\Giang\AppData\Local\Temp\ipykernel_8996\1509227337.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Thời gian đăng'] = df['Ngày đăng'].apply(process_ngay_dang)
C:\Users\Giang\AppData\Local\Temp\ipykernel_8996\1509227337.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Thời gian đăng'] = pd.to_datetime(df['Thời gian đăng'], errors='coerce')


,Tên dự án,Giá,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố,Thời gian đăng
0,NHÀ NGUYỄN XIỂN - QUẬN 9 - 654M2 - NGANG 26M -...,"18,2 tỷ",654 m²,4 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư,Quận 9,Hồ Chí Minh,2025-04-02
1,"SIÊU PHẨM ĐẦU TƯ NHÀ ĐẤT THỦ ĐỨC 750M2, MT 23M...",17 tỷ,750 m²,15 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư,Thủ Đức,Hồ Chí Minh,2025-04-15
2,"BÁN NHÀ 3 MẶT, ĐƯỜNG TX22, PHƯỜNG THẠNH XUÂN, ...",16 tỷ,373 m²,NaN,NaN,NaN,Nhà Đất Thổ Cư,Quận 12,Hồ Chí Minh,2025-01-07
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,"2,45 tỷ",61 m²,4 Phòng ngủ,3 WC,NaN,Nhà Đất Thổ Cư,Quận 3,Hồ Chí Minh,2025-06-03
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,"4,95 tỷ",50 m²,1 Phòng ngủ,1 WC,NaN,Nhà Đất Thổ Cư,Quận 2,Hồ Chí Minh,2025-03-14
...,...,...,...,...,...,...,...,...,...,...
614376,48M2 ĐA TỐN NGÕ 2.5M GIÁ CHỈ 1.95 TỶ BỚT NHIỀU...,1.95 Tỷ,48 M²,,,"15/6/2024, lúc: 14 giờ 21 phút",Nhà Đất Thổ Cư,Huyện Gia Lâm,Hà Nội,2024-06-15
614377,-B.Á.N NHÀ BẠCH MAI -- NHÀ ĐẸP LONG LANH- FULL...,4.8 Tỷ,42 M²,,,"15/6/2024, lúc: 14 giờ 20 phút",Nhà Đất Thổ Cư,Quận Hai Bà Trưng,Hà Nội,2024-06-15
614378,👉-B.Á.N ĐỊNH CÔNG THƯỢNG -DT:48MX4TẦNG - GIÁ 6...,6.6 Tỷ,48 M²,,,"15/6/2024, lúc: 14 giờ 16 phút",Nhà Đất Thổ Cư,Quận Hoàng Mai,Hà Nội,2024-06-15
614382,-B.Á.N NHÀ HOÀNG MAI - NHÀ ĐẸP LONG LANH- Ô TÔ...,6.8 Tỷ,40 M²,,,"15/6/2024, lúc: 14 giờ 11 phút",Nhà Đất Thổ Cư,Quận Hoàng Mai,Hà Nội,2024-06-15


In [120]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 351633 entries, 0 to 614383
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Tên dự án       351633 non-null  object        
 1   Giá             351633 non-null  object        
 2   Diện tích       351633 non-null  object        
 3   Phòng ngủ       292954 non-null  object        
 4   Nhà vệ sinh     282107 non-null  object        
 5   Ngày đăng       280401 non-null  object        
 6   Loại BĐS        351633 non-null  object        
 7   Quận/Huyện      351633 non-null  object        
 8   Tỉnh/Thành phố  351633 non-null  object        
 9   Thời gian đăng  351633 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(9)
memory usage: 29.5+ MB


In [121]:
# Lọc chỉ giữ lại các dòng có chứa chữ "tỷ" hoặc "triệu" trong cột "Giá"
df = df[df['Giá'].astype(str).str.contains('tỷ|triệu', case=False, na=False)]

# Loại bỏ các dòng trùng lặp
df.drop_duplicates(inplace=True)

# Tách phần số từ chuỗi trong cột "Giá", giữ lại số có dấu phẩy (phân cách thập phân)
df['Giá (tỷ đồng)'] = df['Giá'].apply(lambda x: re.match(r'[\d,]+', x).group(0) if re.match(r'[\d,]+', x) else None)

# Tách phần chữ (đơn vị "tỷ", "triệu",...) từ chuỗi sau khi loại bỏ phần số
df['dvt'] = df['Giá'].apply(lambda x: re.sub(r'^[\d,]+\s*', '', x))

# Chuyển phần số từ định dạng có dấu phẩy sang dạng float (chuẩn Python)
df['Giá (tỷ đồng)'] = df['Giá (tỷ đồng)'].str.replace(',', '.', regex=False).astype(float)

# Với đơn vị là "triệu", chia cho 1000 để quy đổi về đơn vị tỷ đồng
df.loc[df['dvt'].str.lower() == 'triệu', 'Giá (tỷ đồng)'] = df.loc[df['dvt'].str.lower() == 'triệu', 'Giá (tỷ đồng)'] / 1000

# Lọc giá trị "Giá (tỷ đồng)" trong khoảng từ 0.3 tỷ đến dưới 300 tỷ để loại bỏ ngoại lệ
df = df[(df['Giá (tỷ đồng)'] > 0.2) & (df['Giá (tỷ đồng)'] < 300)]

C:\Users\Giang\AppData\Local\Temp\ipykernel_8996\3168340144.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)
C:\Users\Giang\AppData\Local\Temp\ipykernel_8996\3168340144.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Giá (tỷ đồng)'] = df['Giá'].apply(lambda x: re.match(r'[\d,]+', x).group(0) if re.match(r'[\d,]+', x) else None)
C:\Users\Giang\AppData\Local\Temp\ipykernel_8996\3168340144.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_index

In [122]:
df

,Tên dự án,Giá,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố,Thời gian đăng,Giá (tỷ đồng),dvt
0,NHÀ NGUYỄN XIỂN - QUẬN 9 - 654M2 - NGANG 26M -...,"18,2 tỷ",654 m²,4 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư,Quận 9,Hồ Chí Minh,2025-04-02,18.20,tỷ
1,"SIÊU PHẨM ĐẦU TƯ NHÀ ĐẤT THỦ ĐỨC 750M2, MT 23M...",17 tỷ,750 m²,15 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư,Thủ Đức,Hồ Chí Minh,2025-04-15,17.00,tỷ
2,"BÁN NHÀ 3 MẶT, ĐƯỜNG TX22, PHƯỜNG THẠNH XUÂN, ...",16 tỷ,373 m²,NaN,NaN,NaN,Nhà Đất Thổ Cư,Quận 12,Hồ Chí Minh,2025-01-07,16.00,tỷ
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,"2,45 tỷ",61 m²,4 Phòng ngủ,3 WC,NaN,Nhà Đất Thổ Cư,Quận 3,Hồ Chí Minh,2025-06-03,2.45,tỷ
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,"4,95 tỷ",50 m²,1 Phòng ngủ,1 WC,NaN,Nhà Đất Thổ Cư,Quận 2,Hồ Chí Minh,2025-03-14,4.95,tỷ
...,...,...,...,...,...,...,...,...,...,...,...,...
614374,BÁN CĂN GÓC 63M2 2 NGỦ 2WC CT4 XALA HÀ ĐÔNG CÓ...,2 Tỷ,63 M²,,,"15/6/2024, lúc: 15 giờ 5 phút",Căn Hộ Chung Cư,Quận Hà Đông,Hà Nội,2024-06-15,2.00,Tỷ
614376,48M2 ĐA TỐN NGÕ 2.5M GIÁ CHỈ 1.95 TỶ BỚT NHIỀU...,1.95 Tỷ,48 M²,,,"15/6/2024, lúc: 14 giờ 21 phút",Nhà Đất Thổ Cư,Huyện Gia Lâm,Hà Nội,2024-06-15,1.00,.95 Tỷ
614377,-B.Á.N NHÀ BẠCH MAI -- NHÀ ĐẸP LONG LANH- FULL...,4.8 Tỷ,42 M²,,,"15/6/2024, lúc: 14 giờ 20 phút",Nhà Đất Thổ Cư,Quận Hai Bà Trưng,Hà Nội,2024-06-15,4.00,.8 Tỷ
614378,👉-B.Á.N ĐỊNH CÔNG THƯỢNG -DT:48MX4TẦNG - GIÁ 6...,6.6 Tỷ,48 M²,,,"15/6/2024, lúc: 14 giờ 16 phút",Nhà Đất Thổ Cư,Quận Hoàng Mai,Hà Nội,2024-06-15,6.00,.6 Tỷ


In [123]:
# Loại bỏ các dòng có giá trị thiếu (NaN) trong các cột quan trọng
df.dropna(subset=['Diện tích'], inplace=True)

# Trích xuất diện tích từ chuỗi, chuẩn hóa dấu phẩy sang dấu chấm và chuyển sang float
df['Diện tích(m2)'] = df['Diện tích'].str.extract(r'([\d,.]+)').replace(',', '.', regex=True).astype(float)

# loại bỏ các dòng có diện tích>1000m2 và diện tích<10m2
df = df[(df['Diện tích(m2)'] > 10) & (df['Diện tích(m2)'] < 1000)]

# Tính giá trung bình trên mỗi mét vuông, đơn vị: triệu đồng/m2
df['Triệu/m2'] = round(df['Giá (tỷ đồng)'] * 1000 / df['Diện tích(m2)'], 2)


C:\Users\Giang\AppData\Local\Temp\ipykernel_8996\516864959.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Triệu/m2'] = round(df['Giá (tỷ đồng)'] * 1000 / df['Diện tích(m2)'], 2)


In [124]:
df

,Tên dự án,Giá,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố,Thời gian đăng,Giá (tỷ đồng),dvt,Diện tích(m2),Triệu/m2
0,NHÀ NGUYỄN XIỂN - QUẬN 9 - 654M2 - NGANG 26M -...,"18,2 tỷ",654 m²,4 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư,Quận 9,Hồ Chí Minh,2025-04-02,18.20,tỷ,654.0,27.83
1,"SIÊU PHẨM ĐẦU TƯ NHÀ ĐẤT THỦ ĐỨC 750M2, MT 23M...",17 tỷ,750 m²,15 Phòng ngủ,NaN,NaN,Nhà Đất Thổ Cư,Thủ Đức,Hồ Chí Minh,2025-04-15,17.00,tỷ,750.0,22.67
2,"BÁN NHÀ 3 MẶT, ĐƯỜNG TX22, PHƯỜNG THẠNH XUÂN, ...",16 tỷ,373 m²,NaN,NaN,NaN,Nhà Đất Thổ Cư,Quận 12,Hồ Chí Minh,2025-01-07,16.00,tỷ,373.0,42.90
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,"2,45 tỷ",61 m²,4 Phòng ngủ,3 WC,NaN,Nhà Đất Thổ Cư,Quận 3,Hồ Chí Minh,2025-06-03,2.45,tỷ,61.0,40.16
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,"4,95 tỷ",50 m²,1 Phòng ngủ,1 WC,NaN,Nhà Đất Thổ Cư,Quận 2,Hồ Chí Minh,2025-03-14,4.95,tỷ,50.0,99.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
614374,BÁN CĂN GÓC 63M2 2 NGỦ 2WC CT4 XALA HÀ ĐÔNG CÓ...,2 Tỷ,63 M²,,,"15/6/2024, lúc: 15 giờ 5 phút",Căn Hộ Chung Cư,Quận Hà Đông,Hà Nội,2024-06-15,2.00,Tỷ,63.0,31.75
614376,48M2 ĐA TỐN NGÕ 2.5M GIÁ CHỈ 1.95 TỶ BỚT NHIỀU...,1.95 Tỷ,48 M²,,,"15/6/2024, lúc: 14 giờ 21 phút",Nhà Đất Thổ Cư,Huyện Gia Lâm,Hà Nội,2024-06-15,1.00,.95 Tỷ,48.0,20.83
614377,-B.Á.N NHÀ BẠCH MAI -- NHÀ ĐẸP LONG LANH- FULL...,4.8 Tỷ,42 M²,,,"15/6/2024, lúc: 14 giờ 20 phút",Nhà Đất Thổ Cư,Quận Hai Bà Trưng,Hà Nội,2024-06-15,4.00,.8 Tỷ,42.0,95.24
614378,👉-B.Á.N ĐỊNH CÔNG THƯỢNG -DT:48MX4TẦNG - GIÁ 6...,6.6 Tỷ,48 M²,,,"15/6/2024, lúc: 14 giờ 16 phút",Nhà Đất Thổ Cư,Quận Hoàng Mai,Hà Nội,2024-06-15,6.00,.6 Tỷ,48.0,125.00


In [125]:
# Tách và chuyển đổi số phòng ngủ từ chuỗi sang kiểu float
df['Phòng ngủ'] = df['Phòng ngủ'].str.extract(r'(\d+)').astype(float)

# Tách và chuyển đổi số nhà vệ sinh từ chuỗi sang kiểu float
df['Nhà vệ sinh'] = df['Nhà vệ sinh'].str.extract(r'(\d+)').astype(float)

# Điền giá trị mặc định cho các cột "Phòng ngủ" và "Nhà vệ sinh" nếu chúng có giá trị NaN
df.fillna({
    'Phòng ngủ': 1,
    'Nhà vệ sinh': 1
}, inplace=True)

C:\Users\Giang\AppData\Local\Temp\ipykernel_8996\2377685777.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Phòng ngủ'] = df['Phòng ngủ'].str.extract(r'(\d+)').astype(float)
C:\Users\Giang\AppData\Local\Temp\ipykernel_8996\2377685777.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Nhà vệ sinh'] = df['Nhà vệ sinh'].str.extract(r'(\d+)').astype(float)
C:\Users\Giang\AppData\Local\Temp\ipykernel_8996\2377685777.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from

In [126]:
df

,Tên dự án,Giá,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố,Thời gian đăng,Giá (tỷ đồng),dvt,Diện tích(m2),Triệu/m2
0,NHÀ NGUYỄN XIỂN - QUẬN 9 - 654M2 - NGANG 26M -...,"18,2 tỷ",654 m²,4.0,1.0,NaN,Nhà Đất Thổ Cư,Quận 9,Hồ Chí Minh,2025-04-02,18.20,tỷ,654.0,27.83
1,"SIÊU PHẨM ĐẦU TƯ NHÀ ĐẤT THỦ ĐỨC 750M2, MT 23M...",17 tỷ,750 m²,15.0,1.0,NaN,Nhà Đất Thổ Cư,Thủ Đức,Hồ Chí Minh,2025-04-15,17.00,tỷ,750.0,22.67
2,"BÁN NHÀ 3 MẶT, ĐƯỜNG TX22, PHƯỜNG THẠNH XUÂN, ...",16 tỷ,373 m²,1.0,1.0,NaN,Nhà Đất Thổ Cư,Quận 12,Hồ Chí Minh,2025-01-07,16.00,tỷ,373.0,42.90
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,"2,45 tỷ",61 m²,4.0,3.0,NaN,Nhà Đất Thổ Cư,Quận 3,Hồ Chí Minh,2025-06-03,2.45,tỷ,61.0,40.16
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,"4,95 tỷ",50 m²,1.0,1.0,NaN,Nhà Đất Thổ Cư,Quận 2,Hồ Chí Minh,2025-03-14,4.95,tỷ,50.0,99.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
614374,BÁN CĂN GÓC 63M2 2 NGỦ 2WC CT4 XALA HÀ ĐÔNG CÓ...,2 Tỷ,63 M²,1.0,1.0,"15/6/2024, lúc: 15 giờ 5 phút",Căn Hộ Chung Cư,Quận Hà Đông,Hà Nội,2024-06-15,2.00,Tỷ,63.0,31.75
614376,48M2 ĐA TỐN NGÕ 2.5M GIÁ CHỈ 1.95 TỶ BỚT NHIỀU...,1.95 Tỷ,48 M²,1.0,1.0,"15/6/2024, lúc: 14 giờ 21 phút",Nhà Đất Thổ Cư,Huyện Gia Lâm,Hà Nội,2024-06-15,1.00,.95 Tỷ,48.0,20.83
614377,-B.Á.N NHÀ BẠCH MAI -- NHÀ ĐẸP LONG LANH- FULL...,4.8 Tỷ,42 M²,1.0,1.0,"15/6/2024, lúc: 14 giờ 20 phút",Nhà Đất Thổ Cư,Quận Hai Bà Trưng,Hà Nội,2024-06-15,4.00,.8 Tỷ,42.0,95.24
614378,👉-B.Á.N ĐỊNH CÔNG THƯỢNG -DT:48MX4TẦNG - GIÁ 6...,6.6 Tỷ,48 M²,1.0,1.0,"15/6/2024, lúc: 14 giờ 16 phút",Nhà Đất Thổ Cư,Quận Hoàng Mai,Hà Nội,2024-06-15,6.00,.6 Tỷ,48.0,125.00


In [127]:
danh_sach_quan_huyen = {
    "Hồ Chí Minh": [
        "Quận 1", "Quận 2", "Quận 3", "Quận 4", "Quận 5", "Quận 6", "Quận 7", "Quận 8", "Quận 9", "Quận 10", "Quận 11", "Quận 12",
        "Bình Thạnh", "Phú Nhuận", "Tân Bình", "Tân Phú", "Gò Vấp", "Bình Tân", "Thủ Đức", 
        "Hóc Môn", "Củ Chi", "Bình Chánh", "Nhà Bè", "Cần Giờ"
    ],
    "Hà Nội": [
        "Ba Đình", "Hoàn Kiếm", "Tây Hồ", "Long Biên", "Cầu Giấy", "Đống Đa", "Hai Bà Trưng", "Hoàng Mai", "Thanh Xuân",
        "Sóc Sơn", "Đông Anh", "Gia Lâm", "Nam Từ Liêm", "Bắc Từ Liêm", "Thanh Trì", "Hoài Đức", "Quốc Oai", "Chương Mỹ",
        "Thường Tín", "Phú Xuyên", "Mê Linh", "Hà Đông", "Sơn Tây", "Ba Vì", "Phúc Thọ", "Đan Phượng", "Thạch Thất", "Ứng Hòa", "Mỹ Đức"
    ],
    "Bình Dương": [
        "Thành phố Thủ Dầu Một", "Thành phố Dĩ An", "Thành phố Thuận An", "Thị xã Bến Cát", "Thị xã Tân Uyên",
        "Huyện Bàu Bàng", "Huyện Bắc Tân Uyên", "Huyện Dầu Tiếng", "Huyện Phú Giáo"
    ],
    "Đồng Nai": [
        "Thành phố Biên Hòa", "Thành phố Long Khánh", "Huyện Tân Phú", "Huyện Vĩnh Cửu", "Huyện Định Quán", "Huyện Thống Nhất",
        "Huyện Trảng Bom", "Huyện Cẩm Mỹ", "Huyện Long Thành", "Huyện Nhơn Trạch", "Huyện Xuân Lộc"
    ],
    "Đà Nẵng": [
        "Quận Hải Châu", "Quận Thanh Khê", "Quận Sơn Trà", "Quận Ngũ Hành Sơn", "Quận Liên Chiểu", "Quận Cẩm Lệ",
        "Huyện Hòa Vang", "Huyện Hoàng Sa"
    ]
}

# Tạo từ điển lowercase để dễ kiểm tra
danh_sach_quan_huyen_lower = {
    tinh: [qh.lower() for qh in ds] for tinh, ds in danh_sach_quan_huyen.items()
}

# Hàm chuẩn hóa chuỗi: bỏ dấu, viết thường, xóa khoảng trắng thừa
def normalize_text(text):
    text = unicodedata.normalize('NFKD', text)
    text = ''.join([c for c in text if not unicodedata.combining(c)])
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip()

def chuan_hoa_quan_huyen(row):
    tinh = row['Tỉnh/Thành phố']
    qh_text = str(row['Quận/Huyện'])

    if tinh not in danh_sach_quan_huyen:
        return row['Quận/Huyện']  # Không xử lý tỉnh ngoài danh sách

    norm_qh_text = normalize_text(qh_text)

    for i, qh in enumerate(danh_sach_quan_huyen[tinh]):
        norm_qh = normalize_text(qh)
        # So khớp nguyên từ bằng regex (ngăn quận 1 khớp vào quận 12)
        if re.search(r'\b{}\b'.format(re.escape(norm_qh)), norm_qh_text):
            return danh_sach_quan_huyen[tinh][i]
    return None  # Không khớp


# Tạo cột tạm chứa kết quả
df['Quận/Huyện mới'] = df.apply(chuan_hoa_quan_huyen, axis=1)

# Xoá dòng không khớp
df = df[df['Quận/Huyện mới'].notna()]

# Cập nhật lại cột gốc
df['Quận/Huyện'] = df['Quận/Huyện mới']


C:\Users\Giang\AppData\Local\Temp\ipykernel_8996\3513001707.py:58: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Quận/Huyện mới'] = df.apply(chuan_hoa_quan_huyen, axis=1)
C:\Users\Giang\AppData\Local\Temp\ipykernel_8996\3513001707.py:64: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Quận/Huyện'] = df['Quận/Huyện mới']


In [128]:
# Xóa các cột không còn cần thiết sau xử lý
df.drop(columns=['Giá', 'dvt', 'Quận/Huyện mới'], inplace=True)


C:\Users\Giang\AppData\Local\Temp\ipykernel_8996\2996402630.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=['Giá', 'dvt', 'Quận/Huyện mới'], inplace=True)


In [129]:
# Đổi tên cột
df.rename(columns={'Tên dự án': 'BĐS'}, inplace=True)


C:\Users\Giang\AppData\Local\Temp\ipykernel_8996\2605401157.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={'Tên dự án': 'BĐS'}, inplace=True)


In [130]:
df

,BĐS,Diện tích,Phòng ngủ,Nhà vệ sinh,Ngày đăng,Loại BĐS,Quận/Huyện,Tỉnh/Thành phố,Thời gian đăng,Giá (tỷ đồng),Diện tích(m2),Triệu/m2
0,NHÀ NGUYỄN XIỂN - QUẬN 9 - 654M2 - NGANG 26M -...,654 m²,4.0,1.0,NaN,Nhà Đất Thổ Cư,Quận 9,Hồ Chí Minh,2025-04-02,18.20,654.0,27.83
1,"SIÊU PHẨM ĐẦU TƯ NHÀ ĐẤT THỦ ĐỨC 750M2, MT 23M...",750 m²,15.0,1.0,NaN,Nhà Đất Thổ Cư,Thủ Đức,Hồ Chí Minh,2025-04-15,17.00,750.0,22.67
2,"BÁN NHÀ 3 MẶT, ĐƯỜNG TX22, PHƯỜNG THẠNH XUÂN, ...",373 m²,1.0,1.0,NaN,Nhà Đất Thổ Cư,Quận 12,Hồ Chí Minh,2025-01-07,16.00,373.0,42.90
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,61 m²,4.0,3.0,NaN,Nhà Đất Thổ Cư,Quận 3,Hồ Chí Minh,2025-06-03,2.45,61.0,40.16
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,50 m²,1.0,1.0,NaN,Nhà Đất Thổ Cư,Quận 2,Hồ Chí Minh,2025-03-14,4.95,50.0,99.00
...,...,...,...,...,...,...,...,...,...,...,...,...
614374,BÁN CĂN GÓC 63M2 2 NGỦ 2WC CT4 XALA HÀ ĐÔNG CÓ...,63 M²,1.0,1.0,"15/6/2024, lúc: 15 giờ 5 phút",Căn Hộ Chung Cư,Hà Đông,Hà Nội,2024-06-15,2.00,63.0,31.75
614376,48M2 ĐA TỐN NGÕ 2.5M GIÁ CHỈ 1.95 TỶ BỚT NHIỀU...,48 M²,1.0,1.0,"15/6/2024, lúc: 14 giờ 21 phút",Nhà Đất Thổ Cư,Gia Lâm,Hà Nội,2024-06-15,1.00,48.0,20.83
614377,-B.Á.N NHÀ BẠCH MAI -- NHÀ ĐẸP LONG LANH- FULL...,42 M²,1.0,1.0,"15/6/2024, lúc: 14 giờ 20 phút",Nhà Đất Thổ Cư,Hai Bà Trưng,Hà Nội,2024-06-15,4.00,42.0,95.24
614378,👉-B.Á.N ĐỊNH CÔNG THƯỢNG -DT:48MX4TẦNG - GIÁ 6...,48 M²,1.0,1.0,"15/6/2024, lúc: 14 giờ 16 phút",Nhà Đất Thổ Cư,Hoàng Mai,Hà Nội,2024-06-15,6.00,48.0,125.00


In [131]:
# Sắp xếp lại thứ tự cột để phù hợp với báo cáo/hiển thị
df = df[['BĐS',
 'Loại BĐS',
 'Tỉnh/Thành phố',
 'Quận/Huyện',
 'Diện tích(m2)',
 'Phòng ngủ',
 'Nhà vệ sinh',
 'Giá (tỷ đồng)',
 'Triệu/m2',
 'Thời gian đăng',]]


In [132]:
df

,BĐS,Loại BĐS,Tỉnh/Thành phố,Quận/Huyện,Diện tích(m2),Phòng ngủ,Nhà vệ sinh,Giá (tỷ đồng),Triệu/m2,Thời gian đăng
0,NHÀ NGUYỄN XIỂN - QUẬN 9 - 654M2 - NGANG 26M -...,Nhà Đất Thổ Cư,Hồ Chí Minh,Quận 9,654.0,4.0,1.0,18.20,27.83,2025-04-02
1,"SIÊU PHẨM ĐẦU TƯ NHÀ ĐẤT THỦ ĐỨC 750M2, MT 23M...",Nhà Đất Thổ Cư,Hồ Chí Minh,Thủ Đức,750.0,15.0,1.0,17.00,22.67,2025-04-15
2,"BÁN NHÀ 3 MẶT, ĐƯỜNG TX22, PHƯỜNG THẠNH XUÂN, ...",Nhà Đất Thổ Cư,Hồ Chí Minh,Quận 12,373.0,1.0,1.0,16.00,42.90,2025-01-07
3,CHỈ CẦN 2TỶ450 BẠN ĐÃ CÓ NGAY THU NHẬP 18TRIỆU...,Nhà Đất Thổ Cư,Hồ Chí Minh,Quận 3,61.0,4.0,3.0,2.45,40.16,2025-06-03
4,CĂN HỘ HẠNG SANG THE PRIVÉ 3 MẶT SÔNG - GIÁ TỪ...,Nhà Đất Thổ Cư,Hồ Chí Minh,Quận 2,50.0,1.0,1.0,4.95,99.00,2025-03-14
...,...,...,...,...,...,...,...,...,...,...
614374,BÁN CĂN GÓC 63M2 2 NGỦ 2WC CT4 XALA HÀ ĐÔNG CÓ...,Căn Hộ Chung Cư,Hà Nội,Hà Đông,63.0,1.0,1.0,2.00,31.75,2024-06-15
614376,48M2 ĐA TỐN NGÕ 2.5M GIÁ CHỈ 1.95 TỶ BỚT NHIỀU...,Nhà Đất Thổ Cư,Hà Nội,Gia Lâm,48.0,1.0,1.0,1.00,20.83,2024-06-15
614377,-B.Á.N NHÀ BẠCH MAI -- NHÀ ĐẸP LONG LANH- FULL...,Nhà Đất Thổ Cư,Hà Nội,Hai Bà Trưng,42.0,1.0,1.0,4.00,95.24,2024-06-15
614378,👉-B.Á.N ĐỊNH CÔNG THƯỢNG -DT:48MX4TẦNG - GIÁ 6...,Nhà Đất Thổ Cư,Hà Nội,Hoàng Mai,48.0,1.0,1.0,6.00,125.00,2024-06-15


In [133]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 147232 entries, 0 to 614382
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   BĐS             147232 non-null  object        
 1   Loại BĐS        147232 non-null  object        
 2   Tỉnh/Thành phố  147232 non-null  object        
 3   Quận/Huyện      147232 non-null  object        
 4   Diện tích(m2)   147232 non-null  float64       
 5   Phòng ngủ       147232 non-null  float64       
 6   Nhà vệ sinh     147232 non-null  float64       
 7   Giá (tỷ đồng)   147232 non-null  float64       
 8   Triệu/m2        147232 non-null  float64       
 9   Thời gian đăng  147232 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(5), object(4)
memory usage: 12.4+ MB


In [134]:
df.to_csv('batdongsan_clean.csv', index=False, float_format='%.1f', encoding='utf-8')